# Tutorial 5 — Classificando respostas abertas com um modelo de linguagem

**Nível:** Intermediário · **Tempo:** 40 min

**Pergunta:** Como apoiar a codificação de respostas abertas sobre motivos de migração?

> Dados didáticos sintéticos, não constituem evidência empírica.

## Passo 1 — Preparar exemplos

As respostas são inventadas e não incluem dados pessoais.

In [ ]:
respostas=["Mudei porque consegui um emprego.","Vim morar perto dos meus pais.","Saí por causa de uma enchente.","Vim cursar a universidade.","Procuro trabalho e quero estudar."]
categorias=["trabalho","familia","ambiente","educacao","multiplo","outro"]
for r in respostas:print("-",r)

## Passo 2 — Criar um classificador local de referência

Esta baseline lexical, sem LLM, mostra limitações de vocabulário e dupla motivação.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
exemplos={"trabalho":"emprego trabalho carreira", "familia":"família pais parentes", "ambiente":"enchente seca desastre ambiental", "educacao":"universidade estudar curso educação"}
vet=TfidfVectorizer(strip_accents="unicode",ngram_range=(1,2))
X=vet.fit_transform(list(exemplos.values())+respostas)
sim=cosine_similarity(X[len(exemplos):],X[:len(exemplos)])
for resposta,scores in zip(respostas,sim):
    idx=scores.argmax()
    rotulo=list(exemplos)[idx] if scores[idx]>0 else "outro"
    print(rotulo, "|", resposta)

## Passo 3 — Montar um prompt de classificação

Um prompt especifica classes permitidas, formato e critério de evidência.

In [ ]:
texto=respostas[0]
prompt=f"""Classifique a resposta de pesquisa em uma das categorias: {", ".join(categorias)}.
Retorne JSON com campos categoria, justificativa e trecho_evidencia.
Use apenas o texto recebido. Não invente detalhes.
Resposta: {texto}"""
print(prompt)

## Passo 4 — Executar um LLM opcional pelo Colab

A opção utiliza um modelo pequeno e pode retornar textos que não seguem o formato requerido. Não inserir dados pessoais.

In [ ]:
USAR_LLM=False  # Mude para True somente se optar por baixar modelo externo
if USAR_LLM:
    # Requer internet e download; disponibilidade de recursos pode variar.
    from transformers import pipeline
    gerador=pipeline("text2text-generation",model="google/flan-t5-small",device=-1)
    print(gerador(prompt,max_new_tokens=90)[0]["generated_text"])
else:
    print("Etapa opcional não executada. O notebook funciona sem baixar modelos.")

## Passo 5 — Planejar verificação humana

Definir diretrizes, dupla codificação e revisão de divergências antes de escalar a classificação.

In [ ]:
gabarito_revisado=["trabalho","familia","ambiente","educacao","multiplo"]
print("Categorias de referência didáticas:")
for r,c in zip(respostas,gabarito_revisado):print(c,"|",r)

## Agora é sua vez

Crie outras cinco respostas fictícias e documente como resolver casos ambíguos. Se executar o modelo opcional, compare a saída com a codificação humana.

## Limitações e cuidados

O uso do LLM é opcional e dependente de download. Modelos pequenos podem falhar em português; nenhuma saída substitui protocolo de codificação e validação humana.